# Yönetmelik Asistanı · Colab çalıştırıcı

Bu notebook yalnızca ince bir çalıştırıcıdır (plan 12.2); kodun tamamı depodadır.
VS Code'da Google Colab uzantısıyla açın ve çalışma zamanı olarak **T4 GPU** seçin.

**Faz 0 hedefi:** ortamı kurmak ve `smoke` sağlık kontrolünü geçmek.

## 1 · GPU kontrolü

In [ ]:
!nvidia-smi --query-gpu=name,memory.used,memory.total --format=csv

## 2 · Drive ve önbellek dizinleri

İndeks ve veriler her zaman Drive'da tutulur (oturum düşerse kaybolmasın diye).
Model dosyalarını Drive'a yazmak isteğe bağlıdır: ücretsiz Drive kotası 15 GB'tır ve
Qwen3.5-9B (~6,6 GB) ile BGE-M3 ve reranker (~4,5 GB) birlikte bunun büyük kısmını doldurur.
Varsayılan olarak modeller her oturumda yeniden indirilir.

In [ ]:
import os
from google.colab import drive

drive.mount('/content/drive')

BASE = '/content/drive/MyDrive/yonetmelik-asistani'
MODELLERI_DRIVEDA_TUT = False  # True: indirme süresi kısalır, Drive kotası dolar

os.environ['YA_DATA_DIR'] = f'{BASE}/data'
os.makedirs(os.environ['YA_DATA_DIR'], exist_ok=True)
if MODELLERI_DRIVEDA_TUT:
    os.environ['HF_HOME'] = f'{BASE}/hf-cache'
    os.environ['OLLAMA_MODELS'] = f'{BASE}/ollama-models'
print({k: os.environ.get(k) for k in ('YA_DATA_DIR', 'HF_HOME', 'OLLAMA_MODELS')})

## 3 · Depoyu al ve paketi kur

`REPO_URL` değerini kendi GitHub deponuzla değiştirin. Depo özelse, Colab'ın
*Secrets* bölümüne `GITHUB_TOKEN` ekleyin.

In [ ]:
REPO_URL = 'https://github.com/Cnrygt11/yonetmelik-asistani.git'
REPO_DIR = '/content/yonetmelik-asistani'

try:
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN')
    if token:
        REPO_URL = REPO_URL.replace('https://', f'https://{token}@')
except Exception:
    pass

if os.path.isdir(REPO_DIR):
    !git -C {REPO_DIR} pull --ff-only
else:
    !git clone -q {REPO_URL} {REPO_DIR}
%pip install -q -e {REPO_DIR}

## 4 · Ollama'yı kur, başlat ve modeli indir

In [ ]:
MODEL = 'qwen3.5:9b'  # yedek: 'gemma4:e4b'
!bash {REPO_DIR}/scripts/colab_setup.sh {MODEL}

## 5 · Sağlık kontrolü

Beş soru çalışır: dördü Türkçe akıcılık ve hız, biri boş bağlamda ret davranışı.
Rapor oturum kapanınca kaybolmasın diye Drive'a yazılır. Faz 0 çıkışında bu raporu
depodaki `eval/reports/` klasörüne ekleyin.

In [ ]:
from datetime import datetime
os.makedirs(f'{BASE}/reports', exist_ok=True)
REPORT = f"{BASE}/reports/smoke_{datetime.now():%Y%m%d_%H%M%S}.json"
!cd {REPO_DIR} && python -m yonetmelik_asistani.smoke --model {MODEL} --report {REPORT}
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv